# Valle Alfa · Decisiones de los soldados
Ajusta un DistilBERT en español para producir una orden con destinatario, acción y lugar, o derivar el mensaje a charla. Los nombres se resuelven contra la plantilla viva de Unreal. La misma función de inferencia se utiliza al probar aquí y al jugar.
 
Abre con el kernel **Yell Decisiones CUDA**. Por defecto se comprueban dos minibatches en un directorio de prueba. Activa ENTRENAR para publicar tu propio checkpoint. Las métricas de la corrida corta solo comprueban el circuito.
 
La estructura toma como referencia los cuadernos universitarios de [clasificación](../Tema%20NLP%20Universidad%20codigo%20referencial/18%20nlp/02%20clasificacion%20texto/02%20clasificacion_texto.ipynb) y [fine-tuning](../Tema%20NLP%20Universidad%20codigo%20referencial/18%20nlp/transformers/Copia%20de%20finetune_transformers_pytorch.ipynb).

In [1]:
from pathlib import Path
import os, json, torch
from collections import Counter
if Path('AI NLP/yell').is_dir():
    os.chdir('AI NLP')
from yell.contract import ROOT
from yell.decision import WEIGHTS, DecisionModel, decide
from yell.decision_data import export, read_data
from yell.train_decision import train
from yell.service import active_aliases
ENTRENAR = True
PRUEBA_MINIMA = True
EPOCAS, LOTE, PACIENCIA = 100, 16, 3
prueba = ROOT.parent / '.yell-cache/decision-smoke'
assert torch.cuda.is_available(), 'Selecciona el kernel Yell Decisiones CUDA'
print(torch.__version__, torch.cuda.get_device_name())

c:\Users\aaran\Documents\Semestre 2-26\SIS421\SIS421_02-2026\NLP\Proyecto final\.venv-decision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.6.0+cu124 NVIDIA GeForce RTX 4050 Laptop GPU


## Datos y partición
Cada familia de paráfrasis pertenece a una sola partición. Se reservan nombres nuevos y se incluyen solicitudes incompletas, negativas, preguntas, voseo y ubicaciones contextuales. Al añadir una familia en decision_data.py, activa REGENERAR_DATOS para volver a generar los JSONL. Las órdenes médicas permiten destinatario y paciente distintos. La proporción objetivo 70/15/15 se aplica a las familias por clase. El número de frases únicas puede variar.

In [2]:
REGENERAR_DATOS = False
splits = export() if REGENERAR_DATOS else read_data()
for name, rows in splits.items():
    print(name,len(rows),Counter(r['label'] for r in rows))
for a,b in [('train','val'),('train','test'),('val','test')]:
    assert not {r['family'] for r in splits[a]} & {r['family'] for r in splits[b]}
print(next(r for r in splits['train'] if r['label']=='curar'))

train 1318 Counter({'charla': 248, 'avanzar': 140, 'defender': 130, 'atacar': 130, 'reagruparse': 130, 'patrullar': 130, 'explorar': 130, 'retirarse': 100, 'curar': 90, 'reanimar': 70, 'mantener_posicion': 10, 'cancelar_orden': 10})
val 298 Counter({'charla': 38, 'avanzar': 30, 'defender': 30, 'atacar': 30, 'reagruparse': 30, 'patrullar': 30, 'explorar': 30, 'retirarse': 20, 'curar': 20, 'reanimar': 20, 'mantener_posicion': 10, 'cancelar_orden': 10})
test 314 Counter({'charla': 54, 'avanzar': 30, 'defender': 30, 'atacar': 30, 'retirarse': 30, 'reagruparse': 30, 'patrullar': 30, 'explorar': 30, 'curar': 20, 'mantener_posicion': 10, 'cancelar_orden': 10, 'reanimar': 10})
{'text': 'Alex, traten las lesiones en el flanco izquierdo', 'family': '{s}, traten las lesiones en {z}', 'label': 'curar', 'encoded': 'unidad traten las lesiones en el lugar', 'subject': 'Alfa-1', 'location': 'flanco_izquierdo', 'target': None, 'aliases': {'Alfa-1': ['Alex'], 'Alfa-2': ['Juan'], 'Alfa-3': ['Ana'], 'Alfa

## Ajuste en GPU
Se entrenan los dos últimos bloques y la cabeza de clasificación. AdamW usa tasas distintas para ambos, regularización de pesos, dropout y suavizado de etiquetas. El ciclo incluye warmup lineal, precisión mixta, recorte de gradientes y parada por pérdida de validación. El umbral que envía a charla se elige en validación. La prueba reservada se evalúa una vez sobre el mejor checkpoint.

In [3]:
if ENTRENAR:
    informe = train(WEIGHTS, epochs=EPOCAS, batch_size=LOTE, patience=PACIENCIA)
elif PRUEBA_MINIMA:
    informe = train(prueba, epochs=1, batch_size=4, smoke=True)
else:
    informe = None
if informe:
    print(json.dumps(informe, ensure_ascii=False, indent=2))

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at C:\Users\aaran\Documents\Semestre 2-26\SIS421\SIS421_02-2026\NLP\Proyecto final\AI NLP\models\distilbert-base-es-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'epoch': 1, 'val_loss': 2.416663288270067}
{'epoch': 2, 'val_loss': 2.2146672850487215}
{'epoch': 3, 'val_loss': 1.4779872510257184}
{'epoch': 4, 'val_loss': 1.1779966002342686}
{'epoch': 5, 'val_loss': 1.3352119098573723}
{'epoch': 6, 'val_loss': 1.157815720410955}
{'epoch': 7, 'val_loss': 1.1950415660870954}
{'epoch': 8, 'val_loss': 1.2352483000531294}
{'epoch': 9, 'val_loss': 1.2366058618430322}
{
  "device": "cuda",
  "smoke": false,
  "history": [
    {
      "epoch": 1,
      "val_loss": 2.416663288270067
    },
    {
      "epoch": 2,
      "val_loss": 2.2146672850487215
    },
    {
      "epoch": 3,
      "val_loss": 1.4779872510257184
    },
    {
      "epoch": 4,
      "val_loss": 1.1779966002342686
    },
    {
      "epoch": 5,
      "val_loss": 1.3352119098573723
    },
    {
      "epoch": 6,
      "val_loss": 1.157815720410955
    },
    {
      "epoch": 7,
      "val_loss": 1.1950415660870954
    },
    {
      "epoch": 8,
      "val_loss": 1.2352483000531294
    },


## Probar con tus propias frases
Puedes modificar nombres, sector del jugador y texto. Se imprime el JSON ejecutable y las tres etiquetas más probables. Si la confianza es insuficiente o faltan campos, el resultado es charla. Los pesos de prueba mínima son solo para verificar formatos. El juego carga exclusivamente el checkpoint publicado por el entrenamiento completo.

In [13]:
origen = WEIGHTS if (WEIGHTS/'actual.txt').exists() else prueba
modelo = DecisionModel(origen)
print('Pesos:', origen)
plantilla = {'Alfa-1': {'nombre':'Alex'}, 'Alfa-2': {'nombre':'Juan'}, 'Bravo-1': {'nombre':'Lucía'}}
pelotones = ['Alfa','Bravo']
aliases = {k:v for k,v in active_aliases(pelotones,list(plantilla)).items() if k in pelotones or k in plantilla}
for uid, persona in plantilla.items():
    aliases[uid].append(persona['nombre'])
frase = 'Alex, vamos tu puedes'
sector = 'centro'
decision, alternativas = decide(modelo, frase, aliases, plantilla, sector)
print(json.dumps(decision, ensure_ascii=False, indent=2))
print(alternativas[:3])

Pesos: C:\Users\aaran\Documents\Semestre 2-26\SIS421\SIS421_02-2026\NLP\Proyecto final\AI NLP\models\decisiones
{
  "tipo": "charla",
  "interlocutor": {
    "tipo": "soldado",
    "id": "Alfa-1",
    "nombre": "Alex"
  }
}
[('charla', 0.9796051979064941), ('avanzar', 0.0038902207743376493), ('explorar', 0.003072911873459816)]
